# Taller 05: Clustering DBSCAN

**Datos sintéticos generados a partir de una figura dibujada manualmente en Paint**

En este taller construimos, paso a paso, un flujo completo de clustering basado en densidad (DBSCAN), aplicado sobre un conjunto de datos sintéticos que **no proviene de `make_blobs`, círculos, lunas ni ningún generador artificial de `sklearn.datasets`**, sino de una figura dibujada a mano por el estudiante usando la herramienta de spray/rociador de Microsoft Paint.

El flujo general del taller es:

```
IMAGEN DIBUJADA EN PAINT
        ↓
Procesamiento de imagen
        ↓
Identificación de las zonas pintadas
        ↓
Generación de puntos sintéticos 2D
        ↓
Agregar outliers uniformemente distribuidos
        ↓
Análisis de k-distancia
        ↓
Seleccionar eps y MinPts
        ↓
Implementar DBSCAN
        ↓
Obtener clusters y ruido
        ↓
Evaluar con dos métricas
        ↓
Analizar resultados
```


## 1. Introducción

El objetivo de este taller es comprender e implementar el algoritmo **DBSCAN (Density-Based Spatial Clustering of Applications with Noise)** desde sus fundamentos matemáticos hasta su aplicación práctica sobre un conjunto de datos con formas **amorfas y no convexas**, generado a partir de una imagen dibujada manualmente.

A diferencia de los talleres tradicionales de clustering que usan datasets sintéticos "de juguete" (círculos concéntricos, blobs gaussianos perfectos, lunas), aquí el reto es distinto: los grupos provienen de trazos irregulares de spray hechos a mano, lo que obliga a:

- Procesar una imagen real (con ruido, bordes irregulares y densidad de pintura no uniforme).
- Extraer coordenadas espaciales representativas de las zonas pintadas.
- Generar un dataset sintético 2D que conserve la geometría original de los trazos.
- Enfrentar un escenario realista donde **no existe un número de clusters "correcto" definido de antemano por el código**, sino por la figura misma.

Este enfoque es pedagógicamente más exigente que usar `make_blobs`, porque DBSCAN debe lidiar con formas verdaderamente arbitrarias, tal como ocurre en problemas reales (por ejemplo, detección de regiones geográficas, segmentación de imágenes médicas o agrupamiento de trayectorias GPS).


## 2. Fundamentos de DBSCAN

### ¿Qué es DBSCAN?

DBSCAN es un algoritmo de clustering **basado en densidad**. A diferencia de K-Means, que agrupa los puntos alrededor de centroides y asume clusters de forma aproximadamente esférica, DBSCAN agrupa los puntos que están **densamente conectados entre sí**, sin necesidad de definir de antemano el número de clusters.

### Clustering basado en densidad

La idea central es que un cluster es una región del espacio donde la densidad de puntos es alta, separada de otras regiones por zonas de baja densidad. Esto permite detectar clusters de **forma arbitraria** (amorfa, alargada, en forma de anillo, etc.), algo que K-Means no puede hacer bien.

### Parámetros principales

- **`eps` (ε):** radio de vecindad. Define la distancia máxima entre dos puntos para que se consideren vecinos.
- **`MinPts`:** número mínimo de puntos (incluyéndose a sí mismo) que deben existir dentro del radio `eps` para que un punto se considere un **punto núcleo**.

### Clasificación de puntos

- **Punto núcleo (*core point*):** tiene al menos `MinPts` puntos (incluyéndose a sí mismo) dentro de su vecindad de radio `eps`.
- **Punto frontera (*border point*):** no cumple el criterio de núcleo por sí mismo, pero cae dentro de la vecindad de un punto núcleo.
- **Punto ruido (*noise point*):** no es núcleo ni frontera; no pertenece a ningún cluster. Se etiqueta convencionalmente como `-1`.

### Ventajas de DBSCAN frente a K-Means para formas amorfas

| Característica | K-Means | DBSCAN |
|---|---|---|
| Forma de los clusters | Asume clusters convexos/esféricos | Detecta formas arbitrarias |
| Número de clusters | Debe definirse `k` a priori | Se descubre automáticamente |
| Sensibilidad a outliers | Alta (afectan los centroides) | Baja (los outliers se etiquetan como ruido) |
| Bordes irregulares (spray) | Los distorsiona hacia formas esféricas | Los respeta si la densidad lo permite |

Precisamente porque nuestros datos provienen de trazos de spray con bordes irregulares y formas no convexas, **DBSCAN es la herramienta apropiada**: K-Means fragmentaría o deformaría artificialmente estas figuras para forzarlas a lucir esféricas.


## 3. Figura inicial creada manualmente

La figura utilizada en este taller (`figura_clusters.png`, aproximadamente 550x310 píxeles) fue **dibujada manualmente** por el estudiante en Microsoft Paint, utilizando la herramienta tipo spray/rociador. El resultado son múltiples manchas amorfas sobre fondo blanco, cada una representando un grupo (cluster) que posteriormente se convertirá en datos sintéticos.

> **Nota:** coloca el archivo `figura_clusters.png` en la misma carpeta que este notebook antes de ejecutar la siguiente celda. Si el nombre de tu archivo es distinto, actualiza la variable `RUTA_IMAGEN`.

A continuación se carga y se muestra la imagen original, sin ninguna modificación, para dejar constancia del insumo manual que da origen a todo el taller.


In [ ]:
# Importación de librerías principales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import os

# Semilla global para reproducibilidad de todo el notebook
SEMILLA = 42
np.random.seed(SEMILLA)

# Ruta de la imagen dibujada manualmente en Paint
RUTA_IMAGEN = "figura_clusters.png"

if not os.path.exists(RUTA_IMAGEN):
    raise FileNotFoundError(
        f"No se encontró '{RUTA_IMAGEN}'. Coloca tu imagen dibujada en Paint "
        "en la misma carpeta que este notebook, o actualiza RUTA_IMAGEN."
    )

# Cargamos la imagen original en su formato original (RGB)
imagen_original = Image.open(RUTA_IMAGEN).convert("RGB")
ancho, alto = imagen_original.size
print(f"Dimensiones de la imagen: {ancho} x {alto} px")

plt.figure(figsize=(8, 5))
plt.imshow(imagen_original)
plt.title("Figura original dibujada manualmente en Paint (spray)")
plt.axis("off")
plt.show()

## 4. Procesamiento de la imagen

Para poder extraer los grupos amorfos como datos numéricos, seguimos estos pasos:

1. Convertimos la imagen a **escala de grises**, ya que solo nos interesa distinguir "pintado" de "no pintado" (fondo blanco), no el color exacto usado.
2. Aplicamos un **umbral (threshold)** simple sobre la intensidad de gris para construir una **máscara binaria**: `True` donde hay pintura, `False` donde es fondo blanco.
3. Convertimos los píxeles marcados como `True` en la máscara en **coordenadas `(x, y)`**.

Es importante **no perder la forma original de los trazos**: por eso NO aplicamos suavizados agresivos, ni cierres morfológicos grandes, ni redibujado de las figuras. Usamos directamente los píxeles detectados como base geométrica.


In [ ]:
# Paso 1: conversión a escala de grises
imagen_gris = imagen_original.convert("L")
arreglo_gris = np.array(imagen_gris)

# Paso 2: umbral para construir la mascara binaria
# Usamos un umbral relativamente alto (fondo blanco = 255) para que
# cualquier pixel notablemente mas oscuro que el blanco puro se considere "pintado"
UMBRAL_GRIS = 240
mascara = arreglo_gris < UMBRAL_GRIS  # True = pixel pintado

print(f"Pixeles totales: {mascara.size}")
print(f"Pixeles pintados detectados: {mascara.sum()} "
      f"({100 * mascara.sum() / mascara.size:.2f}% de la imagen)")

plt.figure(figsize=(8, 5))
plt.imshow(mascara, cmap="gray")
plt.title(f"Mascara binaria de zonas pintadas (umbral gris < {UMBRAL_GRIS})")
plt.axis("off")
plt.show()

In [ ]:
# Paso 3: conversion de la mascara a coordenadas (x, y)
# fila (row) = coordenada y en la imagen, columna (col) = coordenada x
filas_pintadas, columnas_pintadas = np.where(mascara)

# Invertimos el eje Y para que la figura se vea con la orientacion
# cartesiana habitual (arriba = valores mayores de y), en lugar de
# la orientacion de imagen (arriba = fila 0)
coord_x = columnas_pintadas.astype(float)
coord_y = (alto - filas_pintadas).astype(float)

coordenadas_pintadas = np.column_stack([coord_x, coord_y])
print(f"Total de coordenadas extraidas de la figura: {coordenadas_pintadas.shape[0]}")

plt.figure(figsize=(8, 5))
plt.scatter(coordenadas_pintadas[:, 0], coordenadas_pintadas[:, 1], s=1, color="black")
plt.title("Coordenadas (x, y) extraidas directamente de la mascara")
plt.xlabel("x")
plt.ylabel("y")
plt.gca().set_aspect("equal", adjustable="box")
plt.show()

## 5. Generación de datos sintéticos

Usar **todos** los píxeles pintados como dataset final no es conveniente: dependiendo de la resolución de la imagen, esto puede producir decenas de miles de puntos casi idénticos (uno por píxel), lo cual no representa un dataset "sintético" propiamente dicho y hace muy lento el clustering.

En su lugar:

1. **Muestreamos aleatoriamente** un subconjunto de las coordenadas pintadas (`N_MUESTRAS_FIGURA` puntos).
2. A cada punto muestreado le sumamos un **ruido gaussiano controlado** (`sigma` pequeño), simulando la variabilidad natural de un proceso de generación de datos reales.
3. Usamos una **semilla fija** (`SEMILLA`) para que el resultado sea reproducible.

Esto conserva la geometría de las manchas de spray (porque partimos de sus coordenadas reales), pero produce un dataset sintético con la variabilidad esperada de datos "ruidosos" del mundo real.


In [ ]:
# Parametros de generacion de datos sinteticos
N_MUESTRAS_FIGURA = 1800   # cantidad de puntos sinteticos a partir de la figura
SIGMA_RUIDO = 1.5          # desviacion estandar del ruido gaussiano (en pixeles)

rng = np.random.default_rng(SEMILLA)

# Muestreo aleatorio sin reemplazo (o con reemplazo si hay pocos pixeles disponibles)
reemplazo = coordenadas_pintadas.shape[0] < N_MUESTRAS_FIGURA
indices_muestra = rng.choice(
    coordenadas_pintadas.shape[0], size=N_MUESTRAS_FIGURA, replace=reemplazo
)
puntos_base = coordenadas_pintadas[indices_muestra]

# Perturbacion gaussiana controlada para simular un dataset sintetico realista
ruido_gaussiano = rng.normal(loc=0.0, scale=SIGMA_RUIDO, size=puntos_base.shape)
datos_sinteticos_figura = puntos_base + ruido_gaussiano

print(f"Datos sinteticos generados a partir de la figura: {datos_sinteticos_figura.shape[0]} puntos")

plt.figure(figsize=(8, 5))
plt.scatter(datos_sinteticos_figura[:, 0], datos_sinteticos_figura[:, 1], s=4, color="steelblue")
plt.title("Datos sinteticos 2D generados a partir de la figura dibujada en Paint")
plt.xlabel("x")
plt.ylabel("y")
plt.gca().set_aspect("equal", adjustable="box")
plt.show()

## 6. Generación de outliers

Para evaluar correctamente la capacidad de DBSCAN de distinguir "ruido" de clusters reales, agregamos un conjunto de **datos atípicos (outliers)** generados de forma **independiente** a los grupos, distribuidos **uniformemente** dentro del rango espacial ocupado por los datos sintéticos de la figura.


In [ ]:
# Parametros de generacion de outliers
PROPORCION_OUTLIERS = 0.08  # proporcion de outliers respecto a los datos de la figura
N_OUTLIERS = int(datos_sinteticos_figura.shape[0] * PROPORCION_OUTLIERS)

# Rango espacial de los datos generados a partir de la figura
x_min, y_min = datos_sinteticos_figura.min(axis=0)
x_max, y_max = datos_sinteticos_figura.max(axis=0)

# Outliers distribuidos uniformemente en el rango espacial del dataset
outliers = rng.uniform(
    low=[x_min, y_min], high=[x_max, y_max], size=(N_OUTLIERS, 2)
)

print(f"Outliers generados: {outliers.shape[0]} "
      f"({PROPORCION_OUTLIERS*100:.1f}% del tamano del dataset de la figura)")

# Dataset final: puntos de la figura + outliers
dataset_completo = np.vstack([datos_sinteticos_figura, outliers])
etiqueta_origen = np.array(
    ["figura"] * datos_sinteticos_figura.shape[0] + ["outlier"] * outliers.shape[0]
)

plt.figure(figsize=(8, 5))
plt.scatter(
    datos_sinteticos_figura[:, 0], datos_sinteticos_figura[:, 1],
    s=4, color="steelblue", label="Puntos de la figura"
)
plt.scatter(
    outliers[:, 0], outliers[:, 1],
    s=12, color="firebrick", marker="x", label="Outliers uniformes"
)
plt.title("Dataset completo: figura original + outliers distribuidos uniformemente")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.gca().set_aspect("equal", adjustable="box")
plt.show()

## 7. Verificación de los grupos

El taller exige que la figura dibujada contenga **al menos 20 grupos amorfos**. En lugar de inventar artificialmente ese número mediante código, verificamos la cantidad real de formas dibujadas en la imagen usando **componentes conectados** sobre la máscara binaria (`scipy.ndimage.label`).

**Limitación esperada:** dado que el spray de Paint deja pequeños huecos y discontinuidades dentro de cada mancha, es posible que el algoritmo de componentes conectados fragmente una misma mancha "visual" en varias regiones separadas, o que dos manchas muy cercanas se detecten como una sola. Para mitigar (sin alterar artificialmente la figura) aplicamos una **dilatación morfológica leve**, que solo cierra huecos pequeños dentro de los mismos trazos, sin fusionar formas que un observador humano consideraría distintas ni inventar geometría nueva.

Este conteo se usa únicamente como **evidencia de verificación** de que la figura maneja como mínimo 20 grupos; no se usa como parámetro de DBSCAN ni condiciona el número de clusters que DBSCAN deberá encontrar más adelante.


In [ ]:
from scipy import ndimage

# Dilatacion leve para cerrar pequenios huecos causados por el spray,
# sin fusionar formas claramente separadas ni inventar geometria
ESTRUCTURA = np.ones((3, 3))  # conectividad 8
mascara_cerrada = ndimage.binary_dilation(mascara, structure=ESTRUCTURA, iterations=2)

# Etiquetado de componentes conectados (cada region = un candidato a grupo)
etiquetas_componentes, numero_componentes = ndimage.label(mascara_cerrada, structure=ESTRUCTURA)

# Filtramos componentes demasiado pequenios (posible ruido de un solo clic del spray)
TAMANIO_MINIMO_COMPONENTE = 15  # pixeles
tamanios = ndimage.sum(mascara_cerrada, etiquetas_componentes, range(1, numero_componentes + 1))
componentes_validos = np.sum(tamanios >= TAMANIO_MINIMO_COMPONENTE)

print(f"Componentes conectados detectados (sin filtrar): {numero_componentes}")
print(f"Componentes conectados validos (tamanio >= {TAMANIO_MINIMO_COMPONENTE} px): {componentes_validos}")

if componentes_validos >= 20:
    print("Requisito cumplido: la figura contiene al menos 20 grupos amorfos.")
else:
    print("ADVERTENCIA: se detectaron menos de 20 componentes validos. "
          "Revisa la figura dibujada o ajusta TAMANIO_MINIMO_COMPONENTE / iterations "
          "de la dilatacion, sin alterar la geometria original de los trazos.")

plt.figure(figsize=(8, 5))
plt.imshow(etiquetas_componentes, cmap="nipy_spectral")
plt.title(f"Componentes conectados detectados en la figura (n = {numero_componentes})")
plt.axis("off")
plt.show()

## 8. Análisis de k-distancia

Para seleccionar un valor razonable de `eps`, usamos el método estándar de **análisis de k-distancia**:

1. Para cada punto, calculamos la distancia a su **k-ésimo vecino más cercano** (con `k` relacionado a `MinPts`).
2. Ordenamos estas distancias de menor a mayor y las graficamos.
3. Buscamos el **"codo" o punto de inflexión** de la curva: representa el valor de distancia a partir del cual la densidad local cambia bruscamente (de zonas densas -clusters- a zonas dispersas -ruido-).
4. Ese valor de distancia en el codo es un buen candidato para `eps`, usando ese mismo `k` como referencia para `MinPts`.

Probamos varios valores de `k` (equivalentes a distintos `MinPts` candidatos) para tener varias referencias antes de decidir.


In [ ]:
from sklearn.neighbors import NearestNeighbors

valores_k_candidatos = [4, 6, 10, 15]

plt.figure(figsize=(9, 6))
for k in valores_k_candidatos:
    vecinos = NearestNeighbors(n_neighbors=k)
    vecinos.fit(dataset_completo)
    distancias, _ = vecinos.kneighbors(dataset_completo)
    # Tomamos la distancia al k-esimo vecino (ultima columna) y la ordenamos
    k_distancias = np.sort(distancias[:, -1])
    plt.plot(k_distancias, label=f"k = {k}")

plt.title("Grafico de k-distancia para distintos valores de k (candidatos a MinPts)")
plt.xlabel("Puntos ordenados por k-distancia")
plt.ylabel("Distancia al k-esimo vecino")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Interpretación del gráfico:**

En cada curva, la mayor parte de los puntos (los que pertenecen a zonas densas, es decir, a los grupos dibujados) tienen una k-distancia baja y relativamente estable. Hacia el extremo derecho de la curva, la distancia crece abruptamente: esos son los puntos con vecinos lejanos, típicamente **outliers** o puntos en los bordes dispersos de las manchas.

El **codo** de cada curva (el punto donde la pendiente cambia de suave a pronunciada) indica el valor de `eps` que separa razonablemente "vecindad densa" de "vecindad dispersa" para ese valor de `k`.

A partir de la inspección visual del gráfico anterior, se toman como candidatos iniciales para la experimentación:

- `MinPts` candidatos: los mismos valores de `k` probados (4, 6, 10, 15).
- `eps` candidato inicial: el valor de distancia donde se observa el codo para `k = MinPts = 10`, que ronda un valor cercano a **eps ≈ 6** (ajustar según el codo real observado en tu ejecución, ya que depende de la escala en píxeles de tu figura).

Estos valores **no se adoptan de forma arbitraria**: se validan formalmente en la sección de Experimentación (sección 10), donde se comparan cuantitativamente distintas combinaciones de `eps` y `MinPts`.


## 9. Implementación propia de DBSCAN

A continuación implementamos DBSCAN **desde cero**, sin depender de `sklearn.cluster.DBSCAN` como solución principal (solo usaremos `NearestNeighbors`/`sklearn.metrics` como utilidades de apoyo, tal como se hizo en el análisis de k-distancia y se hará en la evaluación).

La implementación se divide en tres funciones:

- **`region_query(datos, indice_punto, eps)`**: devuelve los índices de todos los puntos dentro de la vecindad de radio `eps` del punto dado (incluyéndose a sí mismo).
- **`expandir_cluster(...)`**: dado un punto núcleo, expande el cluster incorporando recursivamente/iterativamente a todos los puntos densamente alcanzables.
- **`dbscan(datos, eps, min_pts)`**: función principal que recorre todos los puntos, decide si son núcleo, frontera o ruido, y asigna las etiquetas de cluster (`-1` para ruido).


In [ ]:
from sklearn.neighbors import NearestNeighbors as _NN

def region_query(arbol_vecinos, indice_punto, eps):
    # Devuelve los indices de todos los puntos dentro del radio eps
    # del punto dado (el propio punto queda incluido).
    vecinos = arbol_vecinos.radius_neighbors(
        [datos_para_dbscan[indice_punto]], radius=eps, return_distance=False
    )[0]
    return set(vecinos.tolist())


def expandir_cluster(etiquetas, indice_punto, vecinos, id_cluster, eps, min_pts, arbol_vecinos):
    # Asigna el punto nucleo inicial al cluster actual
    etiquetas[indice_punto] = id_cluster

    # Usamos una lista como cola para procesar los vecinos densamente alcanzables
    cola_vecinos = list(vecinos)
    i = 0
    while i < len(cola_vecinos):
        indice_vecino = cola_vecinos[i]

        if etiquetas[indice_vecino] == -1:
            # Era ruido, ahora se reclasifica como punto frontera de este cluster
            etiquetas[indice_vecino] = id_cluster

        if etiquetas[indice_vecino] == 0:
            # Aun no visitado: se asigna al cluster
            etiquetas[indice_vecino] = id_cluster

            # Se revisa si tambien es un punto nucleo (para seguir expandiendo)
            vecinos_del_vecino = region_query(arbol_vecinos, indice_vecino, eps)
            if len(vecinos_del_vecino) >= min_pts:
                cola_vecinos.extend(vecinos_del_vecino - set(cola_vecinos))

        i += 1

    return etiquetas


def dbscan(datos, eps, min_pts):
    global datos_para_dbscan
    datos_para_dbscan = datos

    n_puntos = datos.shape[0]
    # 0 = no visitado, -1 = ruido, >0 = id de cluster
    etiquetas = np.zeros(n_puntos, dtype=int)

    arbol_vecinos = _NN(radius=eps).fit(datos)

    id_cluster = 0
    for indice_punto in range(n_puntos):
        if etiquetas[indice_punto] != 0:
            continue  # ya fue visitado en una expansion anterior

        vecinos = region_query(arbol_vecinos, indice_punto, eps)

        if len(vecinos) < min_pts:
            etiquetas[indice_punto] = -1  # marcado como ruido (por ahora)
        else:
            id_cluster += 1
            etiquetas = expandir_cluster(
                etiquetas, indice_punto, vecinos, id_cluster, eps, min_pts, arbol_vecinos
            )

    return etiquetas

**Explicación de las funciones:**

- `region_query` usa una estructura de vecinos (`NearestNeighbors` con búsqueda por radio) para encontrar de forma eficiente todos los puntos a una distancia `eps` del punto analizado, incluyéndolo a él mismo. Esto evita calcular manualmente todas las distancias par a par.

- `expandir_cluster` implementa el crecimiento del cluster: parte de un punto núcleo y de su vecindad inicial, y va incorporando iterativamente a todos los puntos alcanzables por densidad. Si encuentra un nuevo punto núcleo dentro de la vecindad, agrega también los vecinos de ese punto a la cola de procesamiento (esto es lo que permite que clusters de forma alargada o amorfa "crezcan" siguiendo la densidad, en vez de limitarse a una vecindad fija).

- `dbscan` es la función orquestadora: recorre cada punto no visitado, decide si es núcleo (si tiene al menos `min_pts` vecinos dentro de `eps`) y, en caso afirmativo, dispara la expansión de un nuevo cluster. Los puntos que nunca alcanzan el umbral de núcleo ni son absorbidos como frontera quedan etiquetados como `-1` (ruido).

Este diseño reproduce fielmente la lógica clásica de DBSCAN (Ester et al., 1996): puntos núcleo, puntos frontera y puntos ruido, sin depender de la implementación interna de `sklearn`.


## 10. Experimentación

Antes de fijar los parámetros finales, exploramos sistemáticamente distintas combinaciones de `eps` y `MinPts`, y registramos para cada combinación:

- Número de clusters encontrados.
- Número de puntos etiquetados como ruido.
- **Silhouette Score** (excluyendo el ruido, ver sección 11).
- **Davies-Bouldin Index** (excluyendo el ruido, ver sección 11).

Esto permite elegir los parámetros finales de forma **justificada y cuantitativa**, y no simplemente porque el resultado se vea "bonito" visualmente.


In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score

def evaluar_configuracion(datos, etiquetas):
    # Calcula Silhouette y Davies-Bouldin excluyendo los puntos de ruido (-1),
    # ya que ambas metricas no estan definidas (o pierden sentido) para el ruido.
    mascara_no_ruido = etiquetas != -1
    etiquetas_validas = etiquetas[mascara_no_ruido]
    datos_validos = datos[mascara_no_ruido]

    n_clusters_validos = len(set(etiquetas_validas))

    # Ambas metricas requieren al menos 2 clusters con puntos asignados
    if n_clusters_validos < 2:
        return np.nan, np.nan

    silhouette = silhouette_score(datos_validos, etiquetas_validas)
    davies_bouldin = davies_bouldin_score(datos_validos, etiquetas_validas)
    return silhouette, davies_bouldin


valores_eps = [3, 4, 5, 6, 7, 8, 10]
valores_min_pts = [4, 6, 8, 10, 12]

resultados_experimentacion = []

for eps in valores_eps:
    for min_pts in valores_min_pts:
        etiquetas = dbscan(dataset_completo, eps=eps, min_pts=min_pts)

        n_clusters = len(set(etiquetas) - {-1})
        n_ruido = int(np.sum(etiquetas == -1))
        silhouette, davies_bouldin = evaluar_configuracion(dataset_completo, etiquetas)

        resultados_experimentacion.append({
            "eps": eps,
            "MinPts": min_pts,
            "n_clusters": n_clusters,
            "n_ruido": n_ruido,
            "Silhouette Score": silhouette,
            "Davies-Bouldin Index": davies_bouldin,
        })

tabla_resultados = pd.DataFrame(resultados_experimentacion)
tabla_resultados.sort_values(["eps", "MinPts"]).reset_index(drop=True)

**Cómo leer la tabla de experimentación:**

- Combinaciones con `eps` muy pequeño tienden a producir muchos clusters diminutos y mucho ruido (porque pocas vecindades alcanzan `MinPts`).
- Combinaciones con `eps` muy grande tienden a fusionar varios grupos en uno solo, reduciendo artificialmente el número de clusters y perdiendo la resolución de las formas dibujadas.
- Un buen equilibrio se refleja en: número de clusters cercano a la cantidad de manchas dibujadas (validada en la sección 7), cantidad de ruido moderada (concentrada en los outliers uniformes, no en los bordes de los grupos), **Silhouette Score alto** (cercano a 1) y **Davies-Bouldin Index bajo** (cercano a 0).

Con base en esta tabla, se selecciona la combinación final de `eps` y `MinPts` para las secciones 11 y 12, priorizando el balance entre las dos métricas y la coherencia con el número de grupos verificado en la sección 7, y no únicamente el aspecto visual del resultado.


## 11. Evaluación

Utilizamos dos métricas estándar de `sklearn.metrics` para evaluar cuantitativamente la calidad del clustering final:

### Silhouette Score

Mide qué tan similar es un punto a los demás puntos de su propio cluster en comparación con los puntos del cluster más cercano. Su rango es `[-1, 1]`:

- Cerca de **+1**: el punto está bien agrupado (cerca de su cluster, lejos de los demás).
- Cerca de **0**: el punto está en el límite entre dos clusters.
- Cerca de **-1**: el punto probablemente fue asignado al cluster equivocado.

**Mejor resultado:** valores más **altos**.

### Davies-Bouldin Index

Mide la relación entre la dispersión interna de los clusters y la separación entre ellos. Compara, para cada par de clusters, la suma de sus dispersiones internas dividida entre la distancia entre sus centroides, y promedia el peor caso para cada cluster.

**Mejor resultado:** valores más **bajos** (cercanos a 0), lo que indica clusters compactos y bien separados entre sí.

### Manejo de los puntos de ruido (`-1`)

Ambas métricas de `sklearn.metrics` **no están diseñadas para incluir la etiqueta de ruido** (`-1`) como si fuera un cluster más: si se incluyera, el ruido —que por definición es disperso y heterogéneo— se trataría como un cluster válido, distorsionando artificialmente ambos puntajes (en particular, penalizando fuertemente el Silhouette Score y el Davies-Bouldin Index de forma no representativa del desempeño real sobre los clusters genuinos).

**Decisión tomada:** para el cálculo final de ambas métricas, se excluyen los puntos etiquetados como `-1` y se evalúan únicamente los puntos asignados a un cluster real. Esta es la misma convención aplicada durante toda la experimentación de la sección 10 (función `evaluar_configuracion`).


In [ ]:
# Seleccion final de parametros, justificada por el analisis de k-distancia (seccion 8)
# y validada cuantitativamente mediante la tabla de experimentacion (seccion 10).
# Ajusta estos valores segun los resultados obtenidos en tu propia ejecucion.
EPS_FINAL = 6
MIN_PTS_FINAL = 10

etiquetas_finales = dbscan(dataset_completo, eps=EPS_FINAL, min_pts=MIN_PTS_FINAL)

n_clusters_finales = len(set(etiquetas_finales) - {-1})
n_ruido_finales = int(np.sum(etiquetas_finales == -1))

silhouette_final, davies_bouldin_final = evaluar_configuracion(dataset_completo, etiquetas_finales)

print(f"Parametros finales: eps = {EPS_FINAL}, MinPts = {MIN_PTS_FINAL}")
print(f"Numero de clusters encontrados: {n_clusters_finales}")
print(f"Numero de puntos de ruido: {n_ruido_finales} de {dataset_completo.shape[0]} "
      f"({100 * n_ruido_finales / dataset_completo.shape[0]:.2f}%)")
print(f"Silhouette Score (sin ruido): {silhouette_final:.4f}")
print(f"Davies-Bouldin Index (sin ruido): {davies_bouldin_final:.4f}")

## 12. Resultado final

A continuación se genera la visualización final del clustering, con:

- Cada cluster identificado con un color diferente.
- Los puntos de ruido marcados con una `x`.
- Título indicando `MinPts`, `eps` y número de clusters encontrados.
- Ejes `x` e `y` etiquetados.
- Leyenda distinguiendo el ruido.


In [ ]:
plt.figure(figsize=(10, 7))

ids_clusters_unicos = sorted(set(etiquetas_finales) - {-1})
mapa_colores = plt.colormaps.get_cmap("tab20").resampled(max(len(ids_clusters_unicos), 1))

for posicion, id_cluster in enumerate(ids_clusters_unicos):
    puntos_cluster = dataset_completo[etiquetas_finales == id_cluster]
    plt.scatter(
        puntos_cluster[:, 0], puntos_cluster[:, 1],
        s=8, color=mapa_colores(posicion), label=f"Cluster {id_cluster}"
    )

puntos_ruido = dataset_completo[etiquetas_finales == -1]
plt.scatter(
    puntos_ruido[:, 0], puntos_ruido[:, 1],
    s=20, c="black", marker="x", label="Ruido"
)

plt.title(
    f"DBSCAN (implementacion propia) | eps = {EPS_FINAL}, MinPts = {MIN_PTS_FINAL} | "
    f"Clusters encontrados: {n_clusters_finales}"
)
plt.xlabel("x")
plt.ylabel("y")
plt.gca().set_aspect("equal", adjustable="box")

# La leyenda puede tener muchas entradas si hay muchos clusters; se limita
# el numero de columnas para que sea legible
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=6, fontsize=8)
plt.tight_layout()
plt.show()

## 13. Análisis de resultados

*(Completa esta sección con los valores numéricos concretos que obtuviste al ejecutar tu notebook, ya que dependen de tu figura específica.)*

- **Clusters encontrados:** DBSCAN identificó `n_clusters_finales` clusters con los parámetros finales (`eps = EPS_FINAL`, `MinPts = MIN_PTS_FINAL`). Compara este número con el conteo de componentes conectados válidos de la sección 7: si son razonablemente cercanos, es evidencia de que DBSCAN está recuperando fielmente los grupos dibujados a mano.

- **Puntos de ruido:** observa qué porcentaje de `n_ruido_finales` corresponde efectivamente a los outliers uniformes generados en la sección 6, en lugar de a puntos legítimos de los grupos. Un buen resultado se caracteriza porque la mayoría del ruido detectado coincide con los outliers introducidos artificialmente, y no con los bordes de las manchas dibujadas.

- **Efecto de `eps`:** revisando la tabla de la sección 10, describe cómo al aumentar `eps` tienden a fusionarse clusters vecinos (reduciendo `n_clusters` pero también reduciendo `n_ruido`), y cómo al disminuirlo aumentan tanto el número de clusters pequeños como el ruido.

- **Efecto de `MinPts`:** describe cómo valores más altos de `MinPts` exigen vecindades más densas para considerar un punto como núcleo, lo que tiende a aumentar el ruido y a reducir clusters pequeños o dispersos; valores más bajos son más permisivos y tienden a fusionar zonas cercanas.

- **Separación de los clusters:** relaciona el valor final de Silhouette Score obtenido (`silhouette_final`) con la calidad visual observada en la gráfica de la sección 12: valores cercanos a 1 deberían corresponder a clusters visualmente bien delimitados y separados.

- **Compacidad y separación (Davies-Bouldin):** relaciona el valor final de `davies_bouldin_final` con la homogeneidad interna de cada mancha de spray: valores bajos indican que los clusters son internamente compactos y están bien diferenciados entre sí.

- **Pertinencia de DBSCAN para datos amorfos:** justifica, con base en los resultados obtenidos, por qué un algoritmo basado en densidad como DBSCAN es más apropiado que K-Means para este tipo de figuras irregulares generadas a mano, retomando los argumentos de la sección 2 (formas no convexas, número de clusters desconocido a priori, robustez frente a outliers).


## 14. Conclusiones

*(Redacta aquí tus conclusiones académicas finales, basadas en los resultados numéricos y gráficos concretos de tu ejecución. Algunos puntos que conviene abordar:)*

- Qué tan fielmente el pipeline (imagen → máscara → coordenadas → muestreo con ruido gaussiano) logró preservar la geometría original de los trazos de spray dibujados en Paint.
- Qué aprendizajes deja la comparación entre el número de grupos verificado manualmente/mediante componentes conectados (sección 7) y el número de clusters finalmente encontrado por DBSCAN (sección 12).
- Qué tan sensibles resultaron los resultados a la elección de `eps` y `MinPts`, y qué tan útil fue el análisis de k-distancia como punto de partida antes de la experimentación cuantitativa.
- Qué papel jugaron el Silhouette Score y el Davies-Bouldin Index como criterios objetivos para elegir los parámetros finales, en lugar de basarse solo en la inspección visual del scatter plot.
- Reflexión general sobre las ventajas (y limitaciones) de usar datos derivados de un dibujo manual real, en comparación con datasets sintéticos "de juguete" como los generados por `make_blobs`.


## Verificación de requisitos

- [x] Figura creada manualmente.
- [x] Al menos 20 grupos amorfos (verificado mediante componentes conectados en la sección 7).
- [x] Datos sintéticos 2D generados a partir de la figura (sección 5).
- [x] Outliers uniformemente distribuidos (sección 6).
- [x] Análisis de k-distancia (sección 8).
- [x] Selección justificada de `eps` (secciones 8 y 10).
- [x] Selección justificada de `MinPts` (secciones 8 y 10).
- [x] Implementación propia de DBSCAN (sección 9).
- [x] Experimentación con diferentes parámetros (sección 10).
- [x] Silhouette Score (secciones 10 y 11).
- [x] Davies-Bouldin Index (secciones 10 y 11).
- [x] Visualización final (sección 12).
- [x] Análisis de resultados (sección 13).
- [x] Conclusiones (sección 14).

> Recuerda: los casilleros anteriores describen que el **notebook contiene la implementación** de cada requisito; los valores concretos (números de clusters, métricas, interpretación) dependen de tu figura dibujada específica y deben completarse/verificarse tras ejecutar el notebook de principio a fin con tu propio archivo `figura_clusters.png`.
